# Calculating Power Law Distribution of Allan Factor

In [1]:
import os
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import warnings; warnings.filterwarnings('ignore')

In [2]:
DATA_PATH = 'allan_var-difs-multiling.tsv'

PARAMETERS_PATH = 'av-params-difs-multiling.csv'

vis_path = DATA_PATH.replace('.csv', '.html')

In [3]:
subcorpora_col = 'lang'

In [4]:
# to rename the corpus correctly . . . 
def lang(x):
    return x.split('-')[1]

## Looking at Results

In [5]:
results = pd.read_csv(PARAMETERS_PATH)

In [6]:
results.isna().sum()

x_turn_id          0
self               0
conversation_id    0
speaker            0
slope              0
b0                 0
dtype: int64

In [7]:
results.isna().mean()

x_turn_id          0.0
self               0.0
conversation_id    0.0
speaker            0.0
slope              0.0
b0                 0.0
dtype: float64

In [8]:
(results['slope'].abs() == np.inf).sum(), (results['slope'].abs() == np.inf).mean()

(np.int64(4), np.float64(1.4115919934502132e-05))

In [9]:
results.head()

,x_turn_id,self,conversation_id,speaker,slope,b0
0,callfriend-eng-n-callfriend-eng-n-4175-10,True,callfriend-eng-n-callfriend-eng-n-4175,callfriend-eng-n-callfriend-eng-n-4175-M2,-1.198703,-4.597079
1,callfriend-eng-n-callfriend-eng-n-4175-100,True,callfriend-eng-n-callfriend-eng-n-4175,callfriend-eng-n-callfriend-eng-n-4175-M2,0.455656,-8.571409
2,callfriend-eng-n-callfriend-eng-n-4175-1000,True,callfriend-eng-n-callfriend-eng-n-4175,callfriend-eng-n-callfriend-eng-n-4175-M2,-0.005905,-6.205356
3,callfriend-eng-n-callfriend-eng-n-4175-1001,True,callfriend-eng-n-callfriend-eng-n-4175,callfriend-eng-n-callfriend-eng-n-4175-M1,0.758344,-11.280335
4,callfriend-eng-n-callfriend-eng-n-4175-1002,True,callfriend-eng-n-callfriend-eng-n-4175,callfriend-eng-n-callfriend-eng-n-4175-M2,1.226390,-8.060574


Adding Corpus names back in . . .

In [10]:
results[subcorpora_col] = [lang(x) for x in tqdm(results['conversation_id'])]

  0%|          | 0/283368 [00:00<?, ?it/s]

In [11]:
results[subcorpora_col].value_counts()

lang
spa    85151
eng    51180
cro    44703
zho    28349
ko     26109
fra    25912
deu    20274
fin     1662
yid       28
Name: count, dtype: int64

In [12]:
results.head()

,x_turn_id,self,conversation_id,speaker,slope,b0,lang
0,callfriend-eng-n-callfriend-eng-n-4175-10,True,callfriend-eng-n-callfriend-eng-n-4175,callfriend-eng-n-callfriend-eng-n-4175-M2,-1.198703,-4.597079,eng
1,callfriend-eng-n-callfriend-eng-n-4175-100,True,callfriend-eng-n-callfriend-eng-n-4175,callfriend-eng-n-callfriend-eng-n-4175-M2,0.455656,-8.571409,eng
2,callfriend-eng-n-callfriend-eng-n-4175-1000,True,callfriend-eng-n-callfriend-eng-n-4175,callfriend-eng-n-callfriend-eng-n-4175-M2,-0.005905,-6.205356,eng
3,callfriend-eng-n-callfriend-eng-n-4175-1001,True,callfriend-eng-n-callfriend-eng-n-4175,callfriend-eng-n-callfriend-eng-n-4175-M1,0.758344,-11.280335,eng
4,callfriend-eng-n-callfriend-eng-n-4175-1002,True,callfriend-eng-n-callfriend-eng-n-4175,callfriend-eng-n-callfriend-eng-n-4175-M2,1.226390,-8.060574,eng


In [13]:
sel_ = results['slope'].isna() | (results['slope'].abs() == np.inf) 
sel_.sum(), sel_.mean(), (~sel_).sum()

(np.int64(4), np.float64(1.4115919934502132e-05), np.int64(283364))

In [14]:
sel_ = ~sel_

In [15]:
results['self'].value_counts()

self
True    283368
Name: count, dtype: int64

In [16]:
mu, SE = (results['slope'].loc[sel_].mean(), results['slope'].loc[sel_].std()/np.sqrt(sel_.sum()))
mu, SE

(np.float64(-0.10310024183241762), np.float64(0.0018859207359610266))

t-test for good measure.

In [17]:
mu / SE

np.float64(-54.668385508725976)

#### individual corpora statistics

In [18]:
subcorpus_results = []

In [19]:
for corpus in tqdm(results[subcorpora_col].unique()):
    
    res = {'corpus': corpus}
    
    sub = results.loc[
        results[subcorpora_col].isin([corpus])
        & sel_
    ]
    
    # self vals
    sel__ = sub['self']
    mu, se, n = sub['slope'].loc[sel__].mean(), sub['slope'].loc[sel__].std() / np.sqrt(sel__.sum()), sel__.sum()
    res['alpha'] = mu
    res['se'] = se
    res['n'] = n
    res['t'] = 't({})={}'.format(n-1, mu/se)
    
    # save outputs
    subcorpus_results += [res]

subcorpus_results = pd.DataFrame(subcorpus_results)    

  0%|          | 0/9 [00:00<?, ?it/s]

In [20]:
subcorpus_results.head(n=100)

,corpus,alpha,se,n,t
0,eng,-0.101012,0.004454,51180,t(51179)=-22.6805042720217
1,fra,-0.081057,0.005363,25912,t(25911)=-15.11272009581191
2,ko,-0.136730,0.006980,26109,t(26108)=-19.589237364972604
3,spa,-0.101775,0.003371,85151,t(85150)=-30.19380745356543
4,deu,-0.102876,0.008216,20274,t(20273)=-12.52136096320959
5,zho,-0.091821,0.005835,28347,t(28346)=-15.735327028837427
6,cro,-0.108225,0.004455,44701,t(44700)=-24.290635333322648
7,fin,-0.112737,0.036954,1662,t(1661)=-3.0507537305573105
8,yid,0.178718,0.425565,28,t(27)=0.4199556411702853


In [21]:
# reporting['Var'] = reporting.index.values
# 
# with open(lme_results_name.replace('.csv', '2.txt'), 'w') as f:
#     txt =  reporting[['Var', 'coefs', 'stat', 'p']].to_latex(index=False).replace('\\toprule', '\\hline').replace('\\midrule', '\\hline\\hline').replace('\\bottomrule', '\\hline')
# 
#     txt = txt.replace('\\\\', '\\\\\hline').replace('|', '$|$').replace('_delta', ' $\Delta$')
#     f.write(txt)
#     f.close()

## Visualization

In [22]:
import plotly.express as px
import plotly.graph_objects as go

In [23]:
df = pd.read_table(DATA_PATH, sep='\t')

In [24]:
df[subcorpora_col] = [lang(x) for x in tqdm(df['conversation_id'])]

  0%|          | 0/4685496 [00:00<?, ?it/s]

In [25]:
df.head()

,nx,ny,turn_delta,self,groups,x_speaker,corpus,conversation_id,lang,sample_delta,residual,resid,resid_1,resid_2,N,tau,allan_var
0,12.0,7.0,3.0,True,croation-cro-croation-cro-2011_56-5,croation-cro-croation-cro-2011_56-AUM,croation-cro,croation-cro-croation-cro-2011_56,cro,1,0.102238,-0.314801,-0.157667,-0.073849,21,21,6.094507e-06
1,12.0,8.0,7.0,True,croation-cro-croation-cro-2011_56-5,croation-cro-croation-cro-2011_56-AUM,croation-cro,croation-cro-croation-cro-2011_56,cro,2,0.033382,-0.157667,-0.073849,0.000000,21,21,1.126606e-07
2,12.0,12.0,8.0,True,croation-cro-croation-cro-2011_56-5,croation-cro-croation-cro-2011_56-AUM,croation-cro,croation-cro-croation-cro-2011_56,cro,3,0.090274,-0.073849,0.000000,0.028697,21,21,2.311523e-06
3,12.0,5.0,11.0,True,croation-cro-croation-cro-2011_56-5,croation-cro-croation-cro-2011_56-AUM,croation-cro,croation-cro-croation-cro-2011_56,cro,4,0.049741,0.000000,0.028697,-0.037892,21,21,1.029395e-05
4,12.0,9.0,16.0,True,croation-cro-croation-cro-2011_56-5,croation-cro-croation-cro-2011_56-AUM,croation-cro,croation-cro-croation-cro-2011_56,cro,5,0.000002,0.028697,-0.037892,-0.231420,21,21,1.826944e-05


In [26]:
group_columns = ['sample_delta']
value_columns = ['resid']

In [27]:
df_ = df[group_columns+value_columns].groupby(by=group_columns).agg('mean').reset_index()
df_['std'] = df[group_columns+value_columns].groupby(by=group_columns).agg('std').reset_index()[value_columns[0]]
df_['Nk'] = df[group_columns+value_columns].groupby(by=group_columns).agg('count').reset_index()[value_columns[0]]
df_['SE'] = df_['std'] / np.sqrt(df_['Nk'])

df_.shape

(29, 5)

In [28]:
df_ = df_.rename(columns={'resid': 'Δ Hxy'})

In [29]:
df_.head()

,sample_delta,Δ Hxy,std,Nk,SE
0,1,0.152548,0.696713,291071,0.001291
1,2,0.110125,0.610676,287356,0.001139
2,3,0.094221,0.586038,283368,0.001101
3,4,0.081344,0.570082,279443,0.001078
4,5,0.073197,0.557366,275492,0.001062


In [30]:
df_=df_.loc[(df_['sample_delta'] <= 30)]

In [31]:
new_fig = go.Figure()

In [32]:
line_color_code = 'rgba(65, 105, 225, 1.0)'
ribbon_color_code = 'rgba(65, 105, 225, 0.2)'

In [33]:


new_fig.add_trace(
    go.Scatter(
        x=df_['sample_delta'].values,
        y=df_['Δ Hxy'].values,
        mode='markers+lines',
        name='Δ Hxy',
        legendgroup='Δ Hxy',
        showlegend=False,
        line=dict(color=line_color_code)
    )
)

# upper bounds for ribbon
new_fig.add_trace(
    go.Scatter(
        x=df_['sample_delta'].values,
        y=df_['Δ Hxy'].values + (2*df_['SE'].values),
        mode='lines',
        line=dict(color=line_color_code,width=0),
        name='Δ Hxy'+'_upper_bound',
        showlegend=False,
        legendgroup='Δ Hxy',
        fill='tonexty',
        fillcolor=ribbon_color_code,
    ), 
    # row=row, col=col
)

# lower bounds for ribbon
new_fig.add_trace(
    go.Scatter(
        x=df_['sample_delta'].values,
        y=df_['Δ Hxy'].values - (2*df_['SE'].values),
        mode='lines',
        line=dict(color=line_color_code,width=0),
        name='Δ Hxy'+'_lower_bound',
        showlegend=False,
        legendgroup='Δ Hxy',
        fill='tonexty',
        fillcolor=ribbon_color_code,
    ), 
    # row=row, col=col
)

new_fig.add_hline(y=0,line_color='darkgray',line_width=2)

In [34]:
new_fig.update_layout(
    template='plotly_white',
    xaxis=dict(
        title='turn Δ'
    ),
    yaxis=dict(
        title='Δ Hxy (other - self)'
    ),
    margin=dict(
        l=0,
        r=0,
        b=15,
        t=15
    )
)
new_fig.show()

In [35]:
new_fig.write_html('delta-Hxy.html')

### Individual corpora

##### grouping data

In [36]:
group_columns = [subcorpora_col, 'sample_delta']
value_columns = ['resid']

In [37]:
df_ = df[group_columns+value_columns].groupby(by=group_columns).agg('mean').reset_index()
df_['std'] = df[group_columns+value_columns].groupby(by=group_columns).agg('std').reset_index()[value_columns[0]]
df_['Nk'] = df[group_columns+value_columns].groupby(by=group_columns).agg('count').reset_index()[value_columns[0]]
df_['SE'] = df_['std'] / np.sqrt(df_['Nk'])

df_.shape

(219, 6)

In [38]:
df_ = df_.rename(columns={'resid': 'Δ Hxy'})

In [39]:
df_.head()

,lang,sample_delta,Δ Hxy,std,Nk,SE
0,cro,1,0.054168,0.441947,46006,0.002060
1,cro,2,0.039416,0.374097,45338,0.001757
2,cro,3,0.030720,0.351942,44703,0.001665
3,cro,4,0.027078,0.331360,44082,0.001578
4,cro,5,0.025615,0.324465,43451,0.001557


In [40]:
df_=df_.loc[(df_['sample_delta'] <= 20)]

##### making visualization

In [41]:
line_color_code = 'rgba(65, 105, 225, 1.0)'
ribbon_color_code = 'rgba(65, 105, 225, 0.2)'

In [42]:
if not os.path.exists('individual_corpora'):
    os.mkdir('individual_corpora')

In [43]:
for corpus in tqdm(df_[subcorpora_col].unique()):
    sub = df_.loc[df_[subcorpora_col].isin([corpus])]
    
    new_fig = go.Figure()
    
    new_fig.add_trace(
        go.Scatter(
            x=sub['sample_delta'].values,
            y=sub['Δ Hxy'].values,
            mode='markers+lines',
            name='Δ Hxy',
            legendgroup='Δ Hxy',
            showlegend=False,
            line=dict(color=line_color_code)
        )
    )
    
    # upper bounds for ribbon
    new_fig.add_trace(
        go.Scatter(
            x=sub['sample_delta'].values,
            y=sub['Δ Hxy'].values + (2*sub['SE'].values),
            mode='lines',
            line=dict(color=line_color_code,width=0),
            name='Δ Hxy'+'_upper_bound',
            showlegend=False,
            legendgroup='Δ Hxy',
            fill='tonexty',
            fillcolor=ribbon_color_code,
        ), 
        # row=row, col=col
    )
    
    # lower bounds for ribbon
    new_fig.add_trace(
        go.Scatter(
            x=sub['sample_delta'].values,
            y=sub['Δ Hxy'].values - (2*sub['SE'].values),
            mode='lines',
            line=dict(color=line_color_code,width=0),
            name='Δ Hxy'+'_lower_bound',
            showlegend=False,
            legendgroup='Δ Hxy',
            fill='tonexty',
            fillcolor=ribbon_color_code,
        ), 
        # row=row, col=col
    )
    
    new_fig.add_hline(y=0,line_color='darkgray',line_width=2)
    
    new_fig.update_layout(
        template='plotly_white',
        xaxis=dict(
            title='turn Δ'
        ),
        yaxis=dict(
            title='Δ Hxy (other - self)'
        ),
        margin=dict(
            l=0,
            r=0,
            b=15,
            t=15
        )
    )
    
    new_fig.write_html(
        os.path.join('individual_corpora', corpus + '.html')
    )

  0%|          | 0/9 [00:00<?, ?it/s]

### Add figure to subplots instead.

In [44]:
from plotly.subplots import make_subplots
from plotly.offline import plot as img_out

In [69]:
corpora = ['eng', 'deu', 'spa', 'fra', 'ko', 'zho']

In [70]:
k_cols = 2
k_rows = int(np.ceil(len(corpora)/k_cols))

In [71]:
main_fig = make_subplots(
    rows=k_rows,cols=k_cols,
    subplot_titles=corpora
)

In [72]:
positions = [
    [(row+1, col+1) for col in range(k_cols)] for row in range(k_rows)
]
positions = sum(positions,[])

In [73]:
ct = 0
for corpus in tqdm(corpora):
    sub = df_.loc[df_[subcorpora_col].isin([corpus])]

    row, col = positions[ct]

    main_fig.add_trace(
        go.Scatter(
            x=sub['sample_delta'].values,
            y=sub['Δ Hxy'].values,
            mode='markers+lines',
            name='Δ Hxy',
            legendgroup='Δ Hxy',
            showlegend=False,
            line=dict(color=line_color_code)
        ),
        row=row,col=col
    )

    # upper bounds for ribbon
    main_fig.add_trace(
        go.Scatter(
            x=sub['sample_delta'].values,
            y=sub['Δ Hxy'].values + (2*sub['SE'].values),
            mode='lines',
            line=dict(color=line_color_code,width=0),
            name='Δ Hxy'+'_upper_bound',
            showlegend=False,
            legendgroup='Δ Hxy',
            fill='tonexty',
            fillcolor=ribbon_color_code,
        ),
        row=row, col=col
    )

    # lower bounds for ribbon
    main_fig.add_trace(
        go.Scatter(
            x=sub['sample_delta'].values,
            y=sub['Δ Hxy'].values - (2*sub['SE'].values),
            mode='lines',
            line=dict(color=line_color_code,width=0),
            name='Δ Hxy'+'_lower_bound',
            showlegend=False,
            legendgroup='Δ Hxy',
            fill='tonexty',
            fillcolor=ribbon_color_code,
        ),
        row=row, col=col
    )

    main_fig.add_hline(y=0,line_color='darkgray',line_width=2,row=row,col=col)

    ct+=1

  0%|          | 0/6 [00:00<?, ?it/s]

In [74]:
main_fig.update_layout(template='plotly_white')
# main_fig.update_layout(template='plotly_white',height=750,width=500)
main_fig.show()

In [75]:
main_fig.write_html('individual-plots.html')

In [76]:
# main_fig.to_image('individual-plots',format='svg')
main_fig.update_layout(template='plotly_white',height=750,width=500)
img_out(main_fig, image_filename='individual-plots.svg', image='svg')

'temp-plot.html'